In [34]:
import pandas as pd
df=pd.read_csv("/content/IMDB Dataset.csv",encoding='latin1')
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [35]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [36]:
df['sentiment'] = df['sentiment'].map({
    'positive':1,
    'negative':0,
})

In [37]:
import re
negation_words=["not good","not bad","not great","don't like","didn't like","never liked","wasn't good","isn't good","no good"]
def clean_text(text):
  text=text.lower()
  text=re.sub(r'[^a-z0-9\s]','',text)
  #convert negations into single tokens
  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" ","_"))
  return text

In [38]:
df['review']=df['review'].apply(clean_text)

In [39]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(df['review'],df['sentiment'],test_size=0.2,random_state=42)

In [40]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [41]:
vocab_size = 20000
max_len = 250
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")#oov is out of vocabulary
tokenizer.fit_on_texts(x_train)
x_train_seq = tokenizer.texts_to_sequences(x_train)#changes tokens to sequence(numbers)
x_test_seq = tokenizer.texts_to_sequences(x_test)
x_train_pad = pad_sequences(x_train_seq, maxlen=max_len, padding='post')
#3 review nammalk kittiyal first athine tokenize cheyyum pinne athine sequence cheyyum...inorder to perform lstm,gru etc we
#have to make the length of tokenise equal thats why we are performing padding by
#adding zeroes at the end of sentences which have less tokens
x_test_pad = pad_sequences(x_test_seq, maxlen=max_len, padding='post')


In [42]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,Dropout
model = Sequential([
    Embedding(vocab_size, 128),
    LSTM(128, dropout=0.3, recurrent_dropout=0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

In [43]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']

)
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [50]:
history=model.fit(
    x_train_pad,y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 458s 913ms/step - accuracy: 0.7836 - loss: 0.4602 - val_accuracy: 0.8353 - val_loss: 0.4146
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 519s 951ms/step - accuracy: 0.8857 - loss: 0.2905 - val_accuracy: 0.8637 - val_loss: 0.3659
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 503s 954ms/step - accuracy: 0.9211 - loss: 0.2116 - val_accuracy: 0.8754 - val_loss: 0.3781
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 476s 902ms/step - accuracy: 0.9435 - loss: 0.1572 - val_accuracy: 0.8737 - val_loss: 0.3961
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 455s 909ms/step - accuracy: 0.9500 - loss: 0.1419 - val_accuracy: 0.8608 - val_loss: 0.4107


In [52]:
loss,acc = model.evaluate(x_test_pad,y_test)
print("Text Accuracy: ",acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 37s 114ms/step - accuracy: 0.8649 - loss: 0.3807
Text Accuracy:  0.8648999929428101


In [69]:
def predict_sentiment(review):
  review = clean_text(review)
  seq = tokenizer.texts_to_sequences([review])
  padded = pad_sequences(seq, maxlen=max_len, padding='post')
  prediction = model.predict(padded)[0][0]
  print("\nReview",review)
  print("Score", prediction)
  if(prediction >=0.55):
    print("Sentiment: Positive")
  else:
    print("Sentiment: Negative")

In [61]:
predict_sentiment("This movie was absolutely amazing an I loved it very much")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step

Review this movie was absolutely amazing an i loved it very much
Score 0.9929373
Sentiment: Positive


In [70]:
predict_sentiment("this is not good....I didn't like the storyline")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step

Review this is not_goodi didnt like the storyline
Score 0.5145084
Sentiment: Negative
